# Price Contour — Online Solver Demo

End-to-end walkthrough of the Lagrangian dual decomposition solver for insurance price optimisation.

In [ ]:
import time

import polars as pl
import price_contour as pc

print(f"price_contour {pc.__version__}")
print(f"polars {pl.__version__}")

## 1. Generate test data

100K synthetic insurance quotes, each scored at 21 multiplier steps (0.80 → 1.20).

In [ ]:
import math

N_QUOTES = 100_000
N_STEPS = 21
scenario_values = [round(0.80 + 0.02 * j, 2) for j in range(N_STEPS)]

quotes = pl.DataFrame({
    "quote_id": [f"Q{q:07d}" for q in range(N_QUOTES)],
    "_q": pl.Series(range(N_QUOTES), dtype=pl.Float64),
})
steps = pl.DataFrame({
    "scenario_index": pl.Series(range(N_STEPS), dtype=pl.Int32),
    "scenario_value": pl.Series(scenario_values, dtype=pl.Float32),
})

sv = pl.col("scenario_value").cast(pl.Float64)
df = (
    quotes.join(steps, how="cross")
    .with_columns(
        _elas=1.5 + 3.5 * pl.col("_q") / N_QUOTES,
        _base=80.0 + 40.0 * pl.col("_q") / N_QUOTES,
    )
    .with_columns(
        _conv=1.0 / (1.0 + (pl.col("_elas") * (sv - 1.0)).exp()),
    )
    .with_columns(
        expected_income=(pl.col("_base") * sv * pl.col("_conv")).cast(pl.Float32),
        volume=pl.col("_conv").cast(pl.Float32),
        loss_ratio=(0.6 / sv * (1.0 + 0.1 * (sv - 1.0))).cast(pl.Float32),
    )
    .select("quote_id", "scenario_index", "scenario_value", "expected_income", "volume", "loss_ratio")
)

print(f"Shape: {df.shape}  ({df['quote_id'].n_unique():,} quotes × {N_STEPS} steps)")
df.head(10)

In [ ]:
# Look at one quote's trade-off curves
q1 = df.filter(pl.col("quote_id") == "Q0000001")
q1

## 2. Unconstrained solve

Without constraints, each quote picks the multiplier that maximises expected income.

In [ ]:
solver_unc = pc.OnlineOptimiser(objective="expected_income")

t0 = time.perf_counter()
result_unc = solver_unc.solve(df)
elapsed = time.perf_counter() - t0

print(f"Time: {elapsed:.2f}s")
print(f"Total objective: {result_unc.total_objective:,.0f}")
result_unc.dataframe.head(5)

Time: 1.44s
Total objective: 702,030,967


quote_id,optimal_step,optimal_multiplier,optimal_objective
str,i32,f32,f32
"""Q0000001""",0,0.8,256.230804
"""Q0000002""",8,0.96,1219.266846
"""Q0000003""",7,0.94,318.798279
"""Q0000004""",0,0.8,533.850891
"""Q0000005""",7,0.94,529.46759


In [ ]:
# Distribution of chosen multipliers (unconstrained)
result_unc.dataframe["optimal_scenario_value"].describe()

## 3. Single constraint: volume ≥ 90% of baseline

The constraint forces the solver to trade off income for retention.

In [ ]:
solver_vol = pc.OnlineOptimiser(
    objective="expected_income",
    constraints={"volume": {"min": 0.90}},
)

t0 = time.perf_counter()
result_vol = solver_vol.solve(df)
elapsed = time.perf_counter() - t0

print(f"Time: {elapsed:.2f}s, Converged: {result_vol.converged}, Iterations: {result_vol.iterations}")
print(f"Total objective: {result_vol.total_objective:,.0f}")
print(f"Lambda (volume): {result_vol.lambdas['volume']:.4f}")
print(f"Total volume:    {result_vol.total_constraints['volume']:,.0f}")

Time: 0.94s, Converged: True, Iterations: 1
Total objective: 702,030,967
Lambda (volume): 0.0000
Total volume:    609,128


In [ ]:
# Compare multiplier distributions: unconstrained vs constrained
comparison = (
    result_unc.dataframe.select(pl.col("optimal_scenario_value").alias("unconstrained"))
    .hstack(result_vol.dataframe.select(pl.col("optimal_scenario_value").alias("constrained")))
)
print("Unconstrained:")
print(comparison["unconstrained"].describe())
print("\nConstrained (volume ≥ 90%):")
print(comparison["constrained"].describe())

In [ ]:
# How many quotes moved to each step?
step_counts = (
    result_vol.dataframe
    .group_by("optimal_scenario_value")
    .len()
    .sort("optimal_scenario_value")
    .rename({"len": "count"})
)
step_counts

## 4. Two constraints: volume ≥ 90% AND loss ratio ≤ 105%

In [ ]:
solver_2c = pc.OnlineOptimiser(
    objective="expected_income",
    constraints={
        "volume": {"min": 0.90},
        "loss_ratio": {"max": 1.05},
    },
)

t0 = time.perf_counter()
result_2c = solver_2c.solve(df)
elapsed = time.perf_counter() - t0

print(f"Time: {elapsed:.2f}s, Converged: {result_2c.converged}, Iterations: {result_2c.iterations}")
print(f"\nLambdas:")
for name, lam in result_2c.lambdas.items():
    print(f"  {name}: {lam:.4f}")
print(f"\nPortfolio totals:")
print(f"  Objective: {result_2c.total_objective:,.0f}")
for name, val in result_2c.total_constraints.items():
    print(f"  {name}: {val:,.2f}")

Time: 1.76s, Converged: False, Iterations: 50

Lambdas:
  loss_ratio: 52.7634
  volume: 0.0000

Portfolio totals:
  Objective: 701,690,076
  loss_ratio: 720,266.19
  volume: 597,174.18


## 5. Warm-start

Reusing lambdas from a previous solve accelerates convergence.

In [ ]:
# Cold start
t0 = time.perf_counter()
cold = solver_2c.solve(df)
cold_time = time.perf_counter() - t0

# Warm start with previous lambdas
t0 = time.perf_counter()
warm = solver_2c.solve(df, lambdas=cold.lambdas)
warm_time = time.perf_counter() - t0

print(f"Cold start: {cold_time:.2f}s, {cold.iterations} iterations")
print(f"Warm start: {warm_time:.2f}s, {warm.iterations} iterations")
print(f"Speedup: {cold_time / warm_time:.1f}x")

Cold start: 1.62s, 50 iterations
Warm start: 1.55s, 50 iterations
Speedup: 1.0x


## 6. Impact analysis

Compare unconstrained vs constrained results.

In [ ]:
# Join unconstrained and constrained results
impact = (
    result_unc.dataframe
    .select(
        "quote_id",
        pl.col("optimal_scenario_value").alias("mult_unconstrained"),
        pl.col("optimal_objective").alias("obj_unconstrained"),
    )
    .join(
        result_vol.dataframe.select(
            "quote_id",
            pl.col("optimal_scenario_value").alias("mult_constrained"),
            pl.col("optimal_objective").alias("obj_constrained"),
            pl.col("optimal_volume").alias("vol_constrained"),
        ),
        on="quote_id",
    )
    .with_columns(
        (pl.col("mult_constrained") - pl.col("mult_unconstrained")).alias("mult_change"),
    )
)

print("Multiplier change distribution (constrained - unconstrained):")
print(impact["mult_change"].describe())

n_decreased = impact.filter(pl.col("mult_change") < -0.001).shape[0]
n_same = impact.filter(pl.col("mult_change").abs() <= 0.001).shape[0]
n_increased = impact.filter(pl.col("mult_change") > 0.001).shape[0]
print(f"\nQuotes with price decrease: {n_decreased:,}")
print(f"Quotes unchanged:           {n_same:,}")
print(f"Quotes with price increase:  {n_increased:,}")

In [ ]:
# Summary comparison
print(f"{'':30s} {'Unconstrained':>15s} {'Vol ≥ 90%':>15s}")
print(f"{'─' * 62}")
print(f"{'Total expected income':30s} {result_unc.total_objective:>15,.0f} {result_vol.total_objective:>15,.0f}")
print(f"{'Mean multiplier':30s} {impact['mult_unconstrained'].mean():>15.4f} {impact['mult_constrained'].mean():>15.4f}")

                                 Unconstrained       Vol ≥ 90%
──────────────────────────────────────────────────────────────
Total expected income              702,030,967     702,030,967
Mean multiplier                         0.8848          0.8848


## 7. Baselines and uplift

The solver now exposes baseline totals (at multiplier=1.0) directly on the result, making it easy to compute uplift percentages and constraint satisfaction ratios.

In [ ]:
# Baselines are computed at multiplier=1.0 (the "do nothing" point)
print(f"Baseline objective:  {result_2c.baseline_objective:,.0f}")
print(f"Optimised objective: {result_2c.total_objective:,.0f}")
uplift = (result_2c.total_objective - result_2c.baseline_objective) / abs(result_2c.baseline_objective) * 100
print(f"Uplift:              {uplift:+.2f}%")

print(f"\nBaseline constraints:")
for name, baseline in result_2c.baseline_constraints.items():
    total = result_2c.total_constraints[name]
    ratio = total / baseline
    print(f"  {name:12s}  baseline={baseline:>12,.0f}  optimised={total:>12,.0f}  ratio={ratio:.4f}")

Baseline objective:  656,953,466
Optimised objective: 701,690,076
Uplift:              +6.81%

Baseline constraints:
  loss_ratio    baseline=     649,961  optimised=     720,266  ratio=1.1082
  volume        baseline=     499,966  optimised=     597,174  ratio=1.1944


## 8. Convergence history

With `record_history=True`, the solver tracks per-iteration snapshots: lambdas, totals, constraint satisfaction. Useful for diagnostics and logging.

In [ ]:
solver_hist = pc.OnlineOptimiser(
    objective="expected_income",
    constraints={
        "volume": {"min": 0.90},
        "loss_ratio": {"max": 1.05},
    },
    record_history=True,
)

result_hist = solver_hist.solve(df)
print(f"Converged: {result_hist.converged}, Iterations: {result_hist.iterations}")
print(f"History records: {len(result_hist.history)}")

# Each record is a dict with iteration snapshot
rec = result_hist.history[0]
print(f"\nFirst iteration keys: {list(rec.keys())}")
print(f"  iteration:          {rec['iteration']}")
print(f"  total_objective:    {rec['total_objective']:,.0f}")
print(f"  lambdas:            {rec['lambdas']}")
print(f"  total_constraints:  { {k: f'{v:,.0f}' for k, v in rec['total_constraints'].items()} }")
print(f"  max_lambda_change:  {rec['max_lambda_change']:.6f}")
print(f"  all_satisfied:      {rec['all_constraints_satisfied']}")

Converged: False, Iterations: 50
History records: 50

First iteration keys: ['all_constraints_satisfied', 'lambdas', 'total_objective', 'total_constraints', 'iteration', 'max_lambda_change']
  iteration:          0
  total_objective:    702,030,967
  lambdas:            {'volume': 0.0, 'loss_ratio': 7.528086528390343}
  total_constraints:  {'volume': '609,128', 'loss_ratio': '733,288'}
  max_lambda_change:  7.528087
  all_satisfied:      False


## 9. Apply — single-pass with stored lambdas

`ApplyOptimiser` takes pre-computed lambdas and applies them in a single forward pass (no iteration). This is what you'd use in production to score new quotes with previously optimised multipliers.

In [ ]:
# Use lambdas from the 2-constraint solve
print(f"Lambdas from solve: {result_2c.lambdas}")

applier = pc.ApplyOptimiser(
    lambdas=result_2c.lambdas,
    objective="expected_income",
    constraints={
        "volume": {"min": 0.90},
        "loss_ratio": {"max": 1.05},
    },
)

t0 = time.perf_counter()
apply_result = applier.apply(df)
elapsed = time.perf_counter() - t0

print(f"\nApply time: {elapsed:.2f}s (single pass, no iteration)")
print(f"Solve objective: {result_2c.total_objective:,.0f}")
print(f"Apply objective: {apply_result.total_objective:,.0f}")
print(f"Match: {abs(apply_result.total_objective - result_2c.total_objective) < 1}")

print(f"\nApply baselines:")
print(f"  baseline_objective: {apply_result.baseline_objective:,.0f}")
for name, val in apply_result.baseline_constraints.items():
    print(f"  baseline_{name}: {val:,.0f}")

apply_result.dataframe.head(5)

Lambdas from solve: {'volume': 0.0, 'loss_ratio': 52.76342243465737}



Apply time: 0.82s (single pass, no iteration)
Solve objective: 701,690,076
Apply objective: 701,690,076
Match: True

Apply baselines:
  baseline_objective: 656,953,466
  baseline_volume: 499,966
  baseline_loss_ratio: 649,961


quote_id,optimal_step,optimal_multiplier,optimal_objective,optimal_volume,optimal_loss_ratio
str,i32,f32,f32,f32,f32
"""Q0000001""",2,0.84,254.553925,0.64785,0.786635
"""Q0000002""",13,1.06,1216.57251,0.488439,0.608665
"""Q0000003""",14,1.08,317.747284,0.445604,0.624068
"""Q0000004""",0,0.8,533.850891,0.709546,0.750363
"""Q0000005""",7,0.94,529.46759,0.532969,0.587741


## 10. Summary — MLflow-ready output

`solver.summary(result)` packages everything into flat dicts ready for logging. Haute can pass these straight to MLflow with no extra computation.

In [ ]:
import json

s = solver_hist.summary(result_hist)

print("=== params (flat, pass to mlflow.log_params) ===")
for k, v in s["params"].items():
    print(f"  {k}: {v!r}")

print(f"\n=== metrics (flat floats, pass to mlflow.log_metrics) ===")
for k, v in s["metrics"].items():
    print(f"  {k}: {v:.4f}")

print(f"\n=== artifacts ===")
print(f"  lambdas:      {s['artifacts']['lambdas']}")
print(f"  config keys:  {list(s['artifacts']['config'].keys())}")
print(f"  summary keys: {list(s['artifacts']['summary'].keys())}")
print(f"  convergence:  {s['artifacts']['convergence'].shape if s['artifacts']['convergence'] is not None else None}")

=== params (flat, pass to mlflow.log_params) ===
  objective: 'expected_income'
  max_iter: 50
  tolerance: 1e-06
  chunk_size: 500000
  n_quotes: 1000000
  n_steps: 21
  constraints: '{"volume": {"min": 0.9}, "loss_ratio": {"max": 1.05}}'
  multiplier_min: 0.8
  multiplier_max: 1.2

=== metrics (flat floats, pass to mlflow.log_metrics) ===
  total_objective: 701690075.6233
  baseline_objective: 656953466.3077
  iterations: 50.0000
  converged: 0.0000
  uplift_pct: 6.8097
  constraint_volume_total: 597174.1774
  constraint_loss_ratio_total: 720266.1907
  constraint_loss_ratio_baseline: 649960.5433
  constraint_loss_ratio_ratio: 1.1082
  constraint_volume_baseline: 499965.9968
  constraint_volume_ratio: 1.1944
  lambda_loss_ratio: 52.7634
  lambda_volume: 0.0000
  multiplier_mean: 0.9031
  multiplier_std: 0.1052
  multiplier_p5: 0.8000
  multiplier_p25: 0.8200
  multiplier_p50: 0.8600
  multiplier_p75: 0.9600
  multiplier_p95: 1.1400

=== artifacts ===
  lambdas:      {'loss_ratio': 52.

In [ ]:
# The convergence artifact is already a Polars DataFrame, ready for .write_parquet()
s["artifacts"]["convergence"]

iteration,total_objective,max_lambda_change,all_constraints_satisfied,lambda_loss_ratio,lambda_volume,constraint_volume,constraint_loss_ratio
i64,f64,f64,bool,f64,f64,f64,f64
0,7.0203e8,7.528087,false,7.528087,0.0,609128.28886,733287.770953
1,7.0202e8,5.125067,false,12.653154,0.0,607458.995098,731396.237113
2,7.0201e8,4.074547,false,16.727701,0.0,606311.577738,730109.196089
3,7.0200e8,3.453517,false,20.181217,0.0,605397.155218,729094.457003
4,7.0198e8,3.032448,false,23.213665,0.0,604623.593933,728241.865467
…,…,…,…,…,…,…,…
45,7.0138e8,0.716771,false,74.385205,0.0,592323.856838,715282.297666
46,7.0136e8,0.705449,false,75.090654,0.0,592156.705682,715113.067497
47,7.0135e8,0.694505,false,75.785159,0.0,591992.11724,714946.699938


In [ ]:
# The summary artifact — human-readable JSON with everything
print(json.dumps(s["artifacts"]["summary"], indent=2))

{
  "solver_type": "online",
  "objective": "expected_income",
  "n_quotes": 1000000,
  "n_steps": 21,
  "iterations": 50,
  "converged": false,
  "total_objective": 701690075.6232872,
  "baseline_objective": 656953466.3077126,
  "lambdas": {
    "volume": 0.0,
    "loss_ratio": 52.76342243465737
  },
  "constraints": {
    "loss_ratio": {
      "total": 720266.1906793416,
      "baseline": 649960.5432585478,
      "lambda": 52.76342243465737,
      "spec": {
        "max": 1.05
      },
      "ratio_to_baseline": 1.1081691006477403
    },
    "volume": {
      "total": 597174.1774199307,
      "baseline": 499965.9968072772,
      "lambda": 0.0,
      "spec": {
        "min": 0.9
      },
      "ratio_to_baseline": 1.1944295836785166
    }
  },
  "multiplier_distribution": {
    "mean": 0.903076515714407,
    "std": 0.10519935190677643,
    "p5": 0.800000011920929,
    "p25": 0.8199999928474426,
    "p50": 0.8600000143051147,
    "p75": 0.9599999785423279,
    "p95": 1.1399999856948853